In [1]:
%pip install kafka-python-ng
%pip install psycopg2-binary
!pip install kafka-python


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import time
from datetime import datetime
from kafka import KafkaConsumer
import psycopg2

ALERT_THRESHOLD_PERCENT = 0.01

# Database connection setup
db_config = {
    "host": "localhost",
    "database": "projekt",
    "user": "student",
    "password": "student",
    "port": 5432,
}

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()
print("Connected to PostgreSQL database!")

# Initialize Kafka Consumer
consumer = KafkaConsumer(
    "crypto-stream",
    bootstrap_servers=["localhost:9092"],
    auto_offset_reset="latest",
    enable_auto_commit=True,
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
)

# Store price history for 0s window sliding
crypto_history = {}

print("Kafka consumer and analyzer running...", flush=True)
print("-" * 85, flush=True)
print(
    f"{'TIME':<10} | {'SYMBOL':<8} | {'PRICE':<10} | {'30s AVG':<10} | {'DIFF %':<8} | {'STATUS':<15}",
    flush=True,
)
print("-" * 85, flush=True)

try:
    for message in consumer:
        data = message.value
        current_time = time.time()

        raw_timestamp = data.get("timestamp")
        symbol = data.get("symbol")
        price = data.get("price")

        if not symbol or price is None:
            continue

        readable_time = datetime.fromtimestamp(raw_timestamp).strftime(
            "%H:%M:%S"
        )

        # Initialize history for a new coin symbol
        if symbol not in crypto_history:
            crypto_history[symbol] = []

        # Track new incoming price record
        crypto_history[symbol].append((current_time, price))

        # Remove data points older than 30 seconds
        crypto_history[symbol] = [
            item
            for item in crypto_history[symbol]
            if current_time - item[0] <= 30
        ]

        # Get older prices to compute the baseline average
        historical_prices = [
            item[1] for item in crypto_history[symbol] if item[0] < current_time
        ]

        if len(historical_prices) > 0:
            avg_price_30s = sum(historical_prices) / len(historical_prices)
            percentage_diff = (
                abs(price - avg_price_30s) / avg_price_30s * 100
            )
        else:
            avg_price_30s = price
            percentage_diff = 0.0

        # Check if the price change exceeds the alert threshold
        if percentage_diff >= ALERT_THRESHOLD_PERCENT and len(historical_prices) > 0:
            status = "!! ANOMALIA !!"

            # Database Save
            try:
                insert_query = """
                INSERT INTO crypto_alerts (timestamp, symbol, current_price, average_price, change_percent)
                VALUES (%s, %s, %s, %s, %s);
                """
                db_timestamp = datetime.fromtimestamp(raw_timestamp)

                cursor.execute(
                    insert_query,
                    (
                        db_timestamp,
                        symbol,
                        price,
                        avg_price_30s,
                        percentage_diff,
                    ),
                )
                conn.commit()  # Commit transaction to PostgreSQL
                status = "!! ALARM ZAPISANY !!"
            except Exception as db_err:
                conn.rollback()
                status = f"DB ERROR: {db_err}"
        else:
            status = "OK"

        print(
            f"{readable_time:<10} | {symbol:<8} | {price:<10.2f} | {avg_price_30s:<10.2f} | {percentage_diff:<7.3f}% | {status:<15}",
            flush=True,
        )

except KeyboardInterrupt:
    print("\nAnalysis system stopped.")
    cursor.close()
    conn.close()

Connected to PostgreSQL database!
Kafka consumer and analyzer running...
-------------------------------------------------------------------------------------
TIME       | SYMBOL   | PRICE      | 30s AVG    | DIFF %   | STATUS         
-------------------------------------------------------------------------------------
